# Aplicações do Aprendizado Contrastivo

Um modelo que transforma cada exemplo em um embedding, com exemplos da mesma classe próximos, resolve problemas em que as classes não existem no treino. Cada pessoa nova é cadastrada com poucas fotos, e reconhecê-la é comparar embeddings, sem retreinar nada.

Este notebook monta um **controle de acesso por reconhecimento facial** com modelos pré-treinados: um banco de rostos, o cadastro de fotos pela webcam, o reconhecimento ao vivo com uma lista de presença e uma visualização dos embeddings. A parte da webcam precisa rodar na sua máquina, porque o OpenCV acessa a câmera local e abre uma janela própria para o vídeo.

In [ ]:
import os
import time
import urllib.request
import zipfile

import cv2
import matplotlib.pyplot as plt
import mediapipe as mp
import numpy as np
import onnxruntime as ort
from matplotlib.offsetbox import AnnotationBbox, OffsetImage
from mediapipe.tasks.python import BaseOptions, vision
from sklearn.datasets import fetch_lfw_pairs, get_data_home
from sklearn.manifold import TSNE

## Os modelos

O reconhecimento tem três etapas: encontrar o rosto, **alinhá-lo** em uma posição padrão e calcular o embedding. Para encontrar o rosto, usamos o detector de landmarks do MediaPipe, do material anterior. Para o embedding, um **ArcFace** do projeto InsightFace: uma MobileFaceNet de 13 MB, treinada em um grande dataset de rostos, que transforma um rosto em um vetor de 512 números. Os pesos são liberados apenas para pesquisa e uso não comercial.

O ArcFace não foi treinado com a InfoNCE: a perda dele classifica as pessoas do treino com uma margem extra no ângulo entre o embedding e o vetor de cada pessoa. O resultado é o mesmo tipo de espaço, com fotos da mesma pessoa próximas, e a camada de classificação é descartada depois do treino. O modelo vem no formato **ONNX**, um formato padrão para redes treinadas, executado pelo `onnxruntime`.

In [ ]:
def download(url, name):
    path = os.path.join("data", name)
    if not os.path.exists(path):
        os.makedirs("data", exist_ok=True)
        urllib.request.urlretrieve(url, path)
    return path


face_detector = vision.FaceLandmarker.create_from_options(vision.FaceLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=download("https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task", "face.task")),
    num_faces=5,
))

if not os.path.exists("data/buffalo_sc/w600k_mbf.onnx"):
    with zipfile.ZipFile(download("https://github.com/deepinsight/insightface/releases/download/v0.7/buffalo_sc.zip", "buffalo_sc.zip")) as z:
        z.extractall("data/buffalo_sc")
options = ort.SessionOptions()
options.log_severity_level = 3   # esconde um aviso inofensivo sobre o tamanho do lote
arcface = ort.InferenceSession("data/buffalo_sc/w600k_mbf.onnx", options, providers=["CPUExecutionProvider"])

### Alinhamento e embedding

O ArcFace foi treinado com rostos de $112 \times 112$ pixels, com os olhos, a ponta do nariz e os cantos da boca sempre nas mesmas posições. Dos 478 pontos do MediaPipe, usamos esses cinco: o centro das íris (468 e 473), a ponta do nariz (1) e os cantos da boca (61 e 291). A `cv2.estimateAffinePartial2D` encontra a rotação, a escala e a translação que levam os cinco pontos às posições padrão, e a `cv2.warpAffine` aplica essa transformação à imagem. Sem o alinhamento, a rede compararia rostos em poses diferentes e erraria mais.

A `find_faces` devolve os rostos alinhados e a caixa de cada um, a `central_face` escolhe o rosto mais perto do centro, para fotos com mais de uma pessoa, e a `embed` devolve os embeddings normalizados.

In [ ]:
FIVE_POINTS = [468, 473, 1, 61, 291]   # íris, íris, nariz, canto da boca, canto da boca
TEMPLATE = np.array([[38.29, 51.70], [73.53, 51.50], [56.03, 71.74], [41.55, 92.37], [70.73, 92.20]], dtype=np.float32)


def find_faces(image):
    # image: RGB uint8 [H, W, 3]
    H, W = image.shape[:2]
    faces, boxes = [], []
    for landmarks in face_detector.detect(mp.Image(image_format=mp.ImageFormat.SRGB, data=image)).face_landmarks:
        points = np.array([[point.x * W, point.y * H] for point in landmarks], dtype=np.float32)   # [478, 2]
        M, _ = cv2.estimateAffinePartial2D(points[FIVE_POINTS], TEMPLATE)
        faces.append(cv2.warpAffine(image, M, (112, 112)))                  # [112, 112, 3]
        boxes.append(np.r_[points.min(axis=0), points.max(axis=0)].astype(int))   # x0, y0, x1, y1
    return faces, boxes


def central_face(image):
    faces, boxes = find_faces(image)
    if not faces:
        return None
    center = np.array(image.shape[1::-1]) / 2
    return faces[int(np.argmin([np.linalg.norm((box[:2] + box[2:]) / 2 - center) for box in boxes]))]


def embed(faces):
    x = (np.stack(faces).astype(np.float32) - 127.5) / 127.5   # [N, 112, 112, 3], entre -1 e 1
    z = arcface.run(None, {"input.1": x.transpose(0, 3, 1, 2)})[0]   # [N, 512]
    return z / np.linalg.norm(z, axis=1, keepdims=True)

### Escolha do limiar

Duas fotos são da mesma pessoa se a similaridade passa de um limiar. Para escolhê-lo, usamos o **LFW** (*Labeled Faces in the Wild*), o benchmark clássico de verificação facial: mil pares de fotos de celebridades tiradas da internet, metade da mesma pessoa e metade de pessoas diferentes. O `scikit-learn` baixa as fotos, cerca de 230 MB, e pedimos as imagens inteiras para o MediaPipe encontrar os pontos.

In [ ]:
lfw = fetch_lfw_pairs(subset="test", color=True, resize=1.0, slice_=None)   # [1000, 2, 250, 250, 3], entre 0 e 1

similarities, same = [], []
for pair, target in zip(lfw.pairs, lfw.target):
    face_a, face_b = [central_face((image * 255).astype(np.uint8)) for image in pair]
    if face_a is not None and face_b is not None:
        z = embed([face_a, face_b])
        similarities.append(z[0] @ z[1])
        same.append(target == 1)
similarities, same = np.array(similarities), np.array(same)

thresholds = np.linspace(-1, 1, 201)
accuracies = [((similarities > t) == same).mean() for t in thresholds]
face_threshold = thresholds[np.argmax(accuracies)]
print(f"{len(same)} pares, limiar {face_threshold:.2f}, acurácia {max(accuracies):.1%}")

plt.hist(similarities[same], bins=40, alpha=0.6, label="mesma pessoa")
plt.hist(similarities[~same], bins=40, alpha=0.6, label="pessoas diferentes")
plt.axvline(face_threshold, color="black", linestyle="--", label="limiar")
plt.xlabel("similaridade do cosseno")
plt.legend()
plt.show()

O modelo erra menos de 1% dos pares, com fotos de iluminação, pose, idade e qualidade bem variadas. Pessoas diferentes ficam com similaridade perto de 0, com embeddings quase perpendiculares.

## O banco de rostos

O banco guarda as fotos alinhadas de cada pessoa, uma pasta por pessoa em `data/faces`. Guardar as fotos, e não só os embeddings, permite recalcular tudo quando o modelo for trocado por um melhor. Para o banco não começar vazio, cadastramos dez fotos de seis celebridades do LFW, que o `scikit-learn` já baixou.

In [ ]:
FACES_DIR = "data/faces"
CELEBRITIES = ["Angelina_Jolie", "Arnold_Schwarzenegger", "Jennifer_Lopez",
               "Luiz_Inacio_Lula_da_Silva", "Serena_Williams", "Vladimir_Putin"]

lfw_dir = os.path.join(get_data_home(), "lfw_home", "lfw_funneled")
for name in CELEBRITIES:
    folder = os.path.join(FACES_DIR, name)
    if os.path.exists(folder):
        continue
    os.makedirs(folder)
    for file in sorted(os.listdir(os.path.join(lfw_dir, name)))[:10]:
        face = central_face(cv2.cvtColor(cv2.imread(os.path.join(lfw_dir, name, file)), cv2.COLOR_BGR2RGB))
        if face is not None:
            cv2.imwrite(os.path.join(folder, file.replace(".jpg", ".png")), cv2.cvtColor(face, cv2.COLOR_RGB2BGR))


def load_database():
    labels, faces = [], []
    for name in sorted(os.listdir(FACES_DIR)):
        for file in sorted(os.listdir(os.path.join(FACES_DIR, name))):
            faces.append(cv2.cvtColor(cv2.imread(os.path.join(FACES_DIR, name, file)), cv2.COLOR_BGR2RGB))
            labels.append(name)
    return np.array(labels), faces, embed(faces)   # [N], N x [112, 112, 3], [N, 512]


labels, faces, embeddings = load_database()
print({name: int((labels == name).sum()) for name in np.unique(labels)})

### Visualização dos embeddings

Para ver os embeddings de 512 dimensões, usamos o **t-SNE**, que projeta os pontos em 2D tentando manter perto os que estão perto no espaço original. As distâncias entre os grupos no gráfico não têm significado; o que importa é quem fica junto de quem. Cada rosto é desenhado na posição do seu embedding, com uma moldura da cor da pessoa, e o `new_face` acrescenta um rosto que não está no banco, em preto.

In [ ]:
def plot_embeddings(new_face=None):
    labels, faces, embeddings = load_database()
    colors = {name: plt.cm.tab10(k % 10) for k, name in enumerate(np.unique(labels))}
    if new_face is not None:
        labels = np.append(labels, "webcam")
        faces = faces + [new_face]
        embeddings = np.vstack([embeddings, embed([new_face])])
        colors["webcam"] = "black"
    points = TSNE(metric="cosine", random_state=0).fit_transform(embeddings)   # [N, 2]

    fig, ax = plt.subplots(figsize=(12, 10))
    for name, color in colors.items():
        ax.scatter(*points[labels == name].T, color=color, label=name)
    for face, point, name in zip(faces, points, labels):
        ax.add_artist(AnnotationBbox(OffsetImage(face, zoom=0.3), point, bboxprops={"edgecolor": colors[name], "linewidth": 3}))
    ax.legend()
    ax.axis("off")
    plt.show()


plot_embeddings()

As fotos de cada pessoa formam um grupo separado, mesmo com poses, idades e iluminações diferentes. A rede nunca foi treinada para separar essas seis pessoas: o espaço já vem organizado assim.

## Cadastro pela webcam

A `run_webcam` é a do material do MediaPipe, com uma diferença: o `process` também recebe a tecla apertada.

A `capture` adiciona fotos de uma pessoa ao banco: aperte **espaço** para tirar uma foto, com um único rosto na imagem, e **q** para terminar. Tire de 5 a 10 fotos girando um pouco a cabeça e mudando a expressão, para cobrir as variações do dia a dia. Rodar de novo com o mesmo nome acrescenta fotos à pessoa, por exemplo com outra iluminação ou de óculos.

In [ ]:
def run_webcam(process, source=0):
    camera = cv2.VideoCapture(source)   # 0 é a webcam; também aceita o caminho de um vídeo
    key = -1
    while True:
        ok, frame = camera.read()
        if not ok:
            break
        frame = cv2.flip(frame, 1)   # espelha na horizontal
        process(frame, key)
        cv2.imshow("webcam", frame)
        key = cv2.waitKey(1) & 0xFF
        if key in (ord("q"), 27):   # q ou Esc
            break
    camera.release()
    cv2.destroyAllWindows()


def draw_box(frame, box, text, color):
    x0, y0, x1, y1 = box
    cv2.rectangle(frame, (x0, y0), (x1, y1), color, 2)
    cv2.putText(frame, text, (x0, y0 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)

In [ ]:
def capture(name):
    folder = os.path.join(FACES_DIR, name)
    os.makedirs(folder, exist_ok=True)
    photos = []

    def process(frame, key):
        faces, boxes = find_faces(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))   # a webcam entrega BGR
        if key == ord(" ") and len(faces) == 1:
            photos.append(faces[0])
            path = os.path.join(folder, f"{len(os.listdir(folder)):03d}.png")
            cv2.imwrite(path, cv2.cvtColor(faces[0], cv2.COLOR_RGB2BGR))
        for box in boxes:
            draw_box(frame, box, f"{name}: {len(photos)} fotos", (0, 200, 255))

    run_webcam(process)
    if photos:
        fig, axes = plt.subplots(1, len(photos), figsize=(2 * len(photos), 2.5), squeeze=False)
        for ax, photo in zip(axes[0], photos):
            ax.imshow(photo)
            ax.axis("off")
        plt.suptitle(f"{name}: rostos alinhados")
        plt.show()


capture("Silvan")   # troque pelo seu nome; rode de novo com outro nome para cadastrar outra pessoa

Os rostos alinhados ficam todos na mesma posição, com os olhos na mesma altura, mesmo que a cabeça estivesse inclinada. No gráfico, as suas fotos devem formar um grupo novo, longe das celebridades.

In [ ]:
plot_embeddings()

## Reconhecimento e lista de presença

O vetor de cada pessoa é a média dos embeddings das suas fotos, normalizada, mais estável que uma foto só. Cada rosto da câmera é comparado com todas as pessoas, um produto de matrizes entre os vetores das pessoas e o embedding do rosto. O rosto recebe o nome da mais similar, se a similaridade passar do limiar; senão, é um desconhecido. A primeira vez que alguém é reconhecido, o horário vai para a lista de presença, e o último rosto visto fica guardado em `latest` para o gráfico.

O limiar veio do LFW, com outra iluminação e outra câmera. Se a mesma pessoa aparece como desconhecida, diminua o `face_threshold`; se pessoas diferentes recebem o mesmo nome, aumente.

In [ ]:
labels, faces, embeddings = load_database()
people = list(np.unique(labels))
centroids = np.stack([embeddings[labels == name].mean(axis=0) for name in people])   # [pessoas, 512]
centroids /= np.linalg.norm(centroids, axis=1, keepdims=True)

attendance = {}   # nome -> horário em que foi reconhecido pela primeira vez
latest = {}       # último rosto visto pela câmera


def process_recognition(frame, key):
    faces, boxes = find_faces(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    if not faces:
        return
    latest["face"] = faces[0]
    similarities = embed(faces) @ centroids.T   # [rostos, pessoas]

    for box, row in zip(boxes, similarities):
        best = int(np.argmax(row))
        if row[best] > face_threshold:
            attendance.setdefault(people[best], time.strftime("%H:%M:%S"))
            draw_box(frame, box, f"{people[best]} {row[best]:.2f}", (0, 200, 0))
        else:
            draw_box(frame, box, f"desconhecido {row[best]:.2f}", (0, 0, 230))


run_webcam(process_recognition)
print(attendance)

O último rosto da câmera, em preto, cai no grupo da pessoa reconhecida. Rode o reconhecimento com outra pessoa na frente da câmera, sem cadastro, e veja onde o rosto dela cai.

In [ ]:
plot_embeddings(latest.get("face"))

Cadastrar alguém é calcular embeddings de algumas fotos, e a rede não muda. É o que um sistema de controle de acesso faz: o servidor guarda as fotos e os vetores, e cada câmera compara os rostos que vê com eles.

Uma foto impressa ou um vídeo no celular também tem o rosto da pessoa, e o modelo os reconhece do mesmo jeito. Experimente com uma foto sua no celular. Sistemas reais acrescentam uma **detecção de vivacidade** (*liveness*), que verifica se o rosto é de uma pessoa presente, por exemplo pedindo para piscar ou usando uma câmera de profundidade.